# 01. Your data

Learn what `forecast()` accepts as input, what it says when the data is wrong, how it reads the frequency and the horizon, and how to leave out days and hours your series never has. Most of the work in a real forecast is getting the data in, so this is the notebook to keep open when something does not behave.

- Input shapes
- Timezones
- Bad data
- Frequency and horizons
- Closed periods with `skip_days` and `skip_hours`

Previous: [00. Quick start](00_quick_start.ipynb) | Next: [02. Trend](02_trend.ipynb)

In [ ]:
import io
import logging
from contextlib import contextmanager

import numpy as np
import pandas as pd

from prophet import forecast
from prophet.quick import (
    DuplicateTimestampsError,
    ForecastInputError,
    horizon_to_periods,
    infer_frequency_details,
    normalise_input,
)

from helpers import load_example, quiet_logs

quiet_logs()

NeoProphet reports every adjustment it makes (dropped rows, a guessed frequency, a removed timezone) through the `prophet.quick` logger. Two small helpers keep that readable here: `show_logs` prints those messages as ordinary output next to the code that caused them, and `attempt` runs a forecast that is expected to fail and prints the error instead of stopping the notebook.

In [ ]:
@contextmanager
def show_logs():
    """Print NeoProphet's log messages as output instead of sending them to stderr."""
    buffer = io.StringIO()
    handler = logging.StreamHandler(buffer)
    handler.setFormatter(logging.Formatter("log: %(message)s"))
    logger = logging.getLogger("prophet.quick")
    logger.addHandler(handler)
    try:
        yield
    finally:
        logger.removeHandler(handler)
        print(buffer.getvalue(), end="")


def attempt(data, **kwargs):
    """Run forecast() and print the ForecastInputError it raises."""
    try:
        forecast(data, **kwargs)
    except ForecastInputError as exc:
        print(f"{type(exc).__name__}: {exc}")

## Input shapes

`forecast()` looks at what you give it, so there is no need to rename columns first. A frame with `ds` and `y` works, and so does a frame with any one date column and any one number column, a frame with a date index, or a plain Series. This section feeds the same monthly series (airline passengers) in five shapes and checks that the forecasts agree.

In [ ]:
df = load_example("air_passengers")
df.head()

In [ ]:
reference = forecast(df, horizon="1 year")
print(reference.summary())
reference.forecast.head()

With other column names, `forecast()` picks the one datetime column and the one numeric column. If there are several candidates it stops and asks, instead of guessing which one you meant.

In [ ]:
renamed = df.rename(columns={"ds": "month", "y": "passengers"})
by_name = forecast(renamed, horizon="1 year")

# A second numeric column (an accidental year column, say) makes the choice ambiguous.
wide = renamed.assign(year=renamed["month"].dt.year)
wide.head(3)

In [ ]:
attempt(wide, horizon="1 year")
explicit = forecast(wide, horizon="1 year", date_col="month", value_col="passengers")

A Series, or a frame with a `DatetimeIndex` and one number column, needs no column names at all.

In [ ]:
series = df.set_index("ds")["y"]
from_series = forecast(series, horizon="1 year")
from_index = forecast(df.set_index("ds"), horizon="1 year")

results = {
    "ds and y columns": reference,
    "other column names": by_name,
    "date_col and value_col": explicit,
    "Series with DatetimeIndex": from_series,
    "frame with DatetimeIndex": from_index,
}
pd.Series(
    {
        name: bool(np.allclose(r.forecast["yhat"], reference.forecast["yhat"]))
        for name, r in results.items()
    },
    name="same forecast as the ds/y frame",
).to_frame()

Takeaway: all five shapes give the same forecast. Rows with missing values are dropped (the log says so) and the rest are put in date order. Here three values are blanked and the rows shuffled; the gaps then make the series irregular, which section "Frequency and horizons" returns to.

In [ ]:
messy = df.sample(frac=1, random_state=0)
messy.loc[messy.index[:3], "y"] = np.nan

with show_logs():
    tidied = forecast(messy, horizon="1 year")
print(tidied.summary())

## Timezones

Prophet works on timestamps without a timezone, so `forecast()` removes the timezone from yours and logs what it did. `tz="wall"` (the default) keeps the clock reading, so midnight stays midnight. `tz="utc"` converts to UTC first. The choice matters around clock changes.

Daily business data usually wants `"wall"`. Take daily Melbourne pedestrian counts around the start of daylight saving on 1 October 2017.

In [ ]:
pedestrians = load_example("pedestrians")
local = pedestrians[pedestrians["ds"].between("2017-09-15", "2017-11-15")].assign(
    ds=lambda d: d["ds"].dt.tz_localize("Australia/Melbourne")
)
local.head(3)

In [ ]:
with show_logs():
    wall = normalise_input(local, tz="wall")
    utc = normalise_input(local, tz="utc")

pd.DataFrame(
    {
        'tz="wall"': wall["ds"].dt.strftime("%H:%M").value_counts(),
        'tz="utc"': utc["ds"].dt.strftime("%H:%M").value_counts(),
    }
).fillna(0).astype(int)

Takeaway: with `"wall"` every day stays at 00:00. With `"utc"` the same days land on two different times of day, 14:00 before the clock change and 13:00 after it, so daily timestamps stop lining up.

Sensor data is the opposite case: each timestamp is an instant, and in wall-clock time the hour when the clocks go back happens twice. Below is a synthetic hourly series (a daily cycle plus noise) across the end of daylight saving on 2 April 2017.

In [ ]:
hours = pd.date_range("2017-03-25", "2017-04-12", freq="h", tz="Australia/Melbourne")
rng = np.random.default_rng(0)
cycle = np.sin(2 * np.pi * hours.tz_convert("UTC").hour / 24)
hourly = pd.Series(50 + 10 * cycle + rng.normal(0, 1, len(hours)), index=hours)

attempt(hourly, horizon="1 day")

The repeated 02:00 appears twice on the same wall clock, so the default fails with a duplicate timestamp error, although the advice in the message (aggregate) is not the best fix here. Converting to UTC gives every reading its own timestamp. The forecast then comes back in UTC, and you convert it back for display.

In [ ]:
with show_logs():
    result = forecast(hourly, horizon="1 day", tz="utc")
print(result.summary())

fcst = result.forecast.assign(
    ds=lambda d: d["ds"].dt.tz_localize("UTC").dt.tz_convert("Australia/Melbourne")
)
fcst.head(3)

## Bad data

Problems with the data raise a specific subclass of `ForecastInputError` before anything is fitted, with a message that says what to do. The examples below break the monthly series from earlier in four ways.

In [ ]:
duplicated = pd.concat([df, df.tail(1)])
attempt(duplicated, horizon="1 year")

In [ ]:
unparseable = df.astype({"ds": str})
unparseable.loc[3, "ds"] = "not a date"
attempt(unparseable, horizon="1 year")

In [ ]:
attempt(df.head(1), horizon="1 year")

In [ ]:
text_values = df.assign(y="n/a")
attempt(text_values, horizon="1 year")

The horizon is checked too. All of these errors inherit from `ForecastInputError`, which is a `ValueError`, so one `except ForecastInputError` clause in your own code catches every one of them.

In [ ]:
attempt(df, horizon="3 fortnights")
DuplicateTimestampsError.__mro__[:3]

## Frequency and horizons

`forecast()` finds the spacing of your data with pandas (`pd.infer_freq`). If the spacing has gaps, it falls back to the most common difference between neighbouring timestamps and logs that it did. A horizon is either a number of periods at that frequency (`30`) or a duration (`"30D"`, `"4 weeks"`, `"2 years"`) that is converted to periods. The conversion functions are plain functions, so the table below needs no model.

In [ ]:
peyton = load_example("peyton_manning")
retail = load_example("retail_sales")
yosemite = load_example("yosemite_temps")

horizons = {
    "daily (Peyton Manning)": (peyton, [30, "30D", "4 weeks", "12 months", "1 year"]),
    "monthly (retail sales)": (retail, [12, "12 months", "1 year", "2 years", "90D"]),
    "5-minute (Yosemite)": (yosemite, [12, "1 hour", "1 day", "2 days"]),
}
pd.DataFrame(
    [
        {
            "data": name,
            "freq": infer_frequency_details(data["ds"]).freq,
            "horizon": horizon,
            "periods": horizon_to_periods(
                horizon,
                data["ds"].max(),
                infer_frequency_details(data["ds"]).freq,
            ),
        }
        for name, (data, options) in horizons.items()
        for horizon in options
    ]
)

A duration counts the steps of the data's own frequency that fall inside it, and always gives at least one. The same `"1 year"` is 366 daily steps for Peyton's data (the year after January 2016 includes a leap day) and 12 monthly steps for retail sales.

### Daily data

The Peyton Manning series is missing 59 days, so pandas finds no exact frequency. NeoProphet uses daily spacing and says so, and the forecast itself is complete. If you know the gaps are expected, pass `freq="D"` and the message goes away.

In [ ]:
found = infer_frequency_details(peyton["ds"])
print(f"freq={found.freq!r}, method={found.method!r}, regular={found.regular}")

with show_logs():
    result = forecast(peyton, horizon="12 months")
print(result.summary())
result.forecast.head(3)

### Monthly data

Monthly dates on the first of each month are recognised as `MS`. `"2 years"` becomes 24 monthly points, and the forecast has one row per month.

`forecast()` only forecasts at the data's own frequency. Monthly data in gives monthly points out, and a horizon shorter than a month still gives one month. The `freq` argument corrects a wrong guess; it does not resample. For daily numbers you need daily data, because a model fitted to monthly totals has no information about the days inside them.

In [ ]:
result = forecast(retail, horizon="2 years")
print(result.summary())
result.forecast.head(3)

In [ ]:
fig = result.plot()

Takeaway: 24 monthly points, one per month, with the yearly pattern carrying on. The band is narrow next to the month-to-month swings in the history, a reminder that intervals often under-cover.

### Sub-daily data

Five-minute temperature readings from Yosemite give 288 points a day, so `"2 days"` is 576 periods. This series has a few blank readings, which are dropped and logged, and the gaps are why the spacing is reported as irregular. This is the largest fit in the notebook, and takes several seconds.

In [ ]:
with show_logs():
    result = forecast(yosemite, horizon="2 days")
print(result.summary())
fig = result.plot()

Takeaway: the 576 points trace the daily temperature cycle. The dotted trend wanders for weeks because it is soaking up weather rather than a lasting trend, so trust this forecast for a day or two, not for weeks.

### Irregular data

Remove a quarter of the months from the retail series. The remaining dates are no longer evenly spaced, but they all fall on the first of a month, so the most common spacing still maps to `MS`.

In [ ]:
sparse = retail.sample(frac=0.75, random_state=1).sort_values("ds")
found = infer_frequency_details(sparse["ds"])
print(
    f"{len(sparse)} of {len(retail)} months kept; freq={found.freq!r}, method={found.method!r}"
)

with show_logs():
    result = forecast(sparse, horizon="1 year")
print(result.summary())

Prophet does not need evenly spaced history, so the fit uses the rows it has. Only the future dates are laid out on the inferred grid. If the guess is wrong, for example when the most common spacing is not the one you want forecast, pass `freq="W"` or another pandas alias to override it.

## Closed periods with `skip_days` and `skip_hours`

Some series have no data at all on certain days or hours: a counter read only on working days, a shop closed at night. Left alone, the model can forecast those closed periods anyway and the charts show empty gaps. `skip_days` and `skip_hours` tell NeoProphet that the periods never exist, so they leave training, the forecast and every chart.

To imitate a counter read only on working days, keep the Monday to Friday rows of the Melbourne pedestrian counts, then take the 12 weeks around Christmas 2020 and remove the public holidays that fall on weekdays, when the counter would also have been off. Twelve weeks is a short window, chosen to make the gaps visible and not as a good amount of training data.

In [ ]:
weekdays = pedestrians[pedestrians["ds"].dt.dayofweek < 5]
holidays = ["01-01", "01-26", "12-25", "12-26"]
window = weekdays[
    weekdays["ds"].between("2020-11-16", "2021-02-12")
    & ~weekdays["ds"].dt.strftime("%m-%d").isin(holidays)
].reset_index(drop=True)

print(
    f"{len(window)} rows, {window['ds'].min():%d %b %Y} to {window['ds'].max():%d %b %Y}"
)
window.assign(day=lambda d: d["ds"].dt.day_name()).iloc[26:32]

Three weekdays are missing, so pandas cannot see a business-day pattern. NeoProphet falls back to daily spacing, and the forecast then has a row for every calendar day, Saturdays and Sundays included, although the series never has one.

In [ ]:
with show_logs():
    plain = forecast(window, horizon="2 weeks")
print(plain.summary())
plain.forecast.assign(day=lambda d: d["ds"].dt.day_name()).head(7)

In [ ]:
fig = plain.plot()

Takeaway: the weekend forecasts dive below zero because the model never saw a weekend, and they drag the headline down to "56% lower". The continuous axis hides that the history has no weekend points either.

With `skip_days="weekends"` the weekend rows are not forecast and the chart closes up, so Friday sits next to Monday. A duration horizon such as `"2 weeks"` is a calendar window with the closed days removed (10 rows); an integer horizon counts rows. The log still notes the three holiday gaps, which `skip_days` does not account for, and that is correct.

In [ ]:
with show_logs():
    closed = forecast(window, horizon="2 weeks", skip_days="weekends")
print(closed.summary())
closed.forecast.assign(day=lambda d: d["ds"].dt.day_name()).head(7)

In [ ]:
fig = closed.plot()

Takeaway: ten plausible weekday forecasts and a headline of about 11% lower. The footnote records that Saturdays and Sundays are left out.

A series that is exactly Monday to Friday with no other gaps is recognised by pandas as business days (`'B'`), so its forecast skips weekends anyway. `skip_days` still closes the chart, and it also covers data that does contain the closed days. Those rows are dropped before fitting and counted in the log and in `summary()`. Here the full seven-day series is given with the same setting, and weekly seasonality is fitted on the five days that remain.

In [ ]:
with show_logs():
    dropped = forecast(pedestrians, horizon="4 weeks", skip_days="weekends")
print(dropped.summary())
fig = dropped.plot_components()

`skip_hours` does the same for hours of the day on sub-daily data. To imitate a sensor that logs nothing between midnight and 08:00, keep only the readings from 08:00 onwards in the last three weeks of the Yosemite series and skip hours 0 to 7. The forecast then covers 16 hours of each day.

In [ ]:
recent = yosemite[yosemite["ds"] >= yosemite["ds"].max() - pd.Timedelta(days=21)]
daytime = recent[recent["ds"].dt.hour >= 8].reset_index(drop=True)

result = forecast(daytime, horizon="2 days", skip_hours=range(0, 8))
print(result.summary())
result.forecast.head(3)

In [ ]:
fig = result.plot()

Takeaway: the line runs from 23:55 straight to 08:00 the next morning, because the night hours are missing from the data and from the forecast. The footnote names what the axis leaves out.

Skipping removes periods, it does not model them: nothing is forecast for the closed times, and `skip_hours` is rejected for data that is not sub-daily.

In [ ]:
attempt(retail, horizon="1 year", skip_hours=range(0, 8))

## Next

Continue with [02. Trend](02_trend.ipynb): how the trend is built from changepoints, how flexible it should be, and how to read what it does to a forecast.